# Prompt Engineering with Azure AI Foundry

Classify news as REAL or FAKE using a hosted GPT-4o model with three prompting
strategies: **zero-shot**, **few-shot (in-context learning)**, and
**chain-of-thought**.

**Prerequisite:** deploy a chat model in Azure AI Foundry and fill `.env`
(see `docs/foundry-setup.md`). This notebook reads the endpoint and key from
`.env` -- no secrets are written into the notebook.

In [1]:
import os, json, time, re
from pathlib import Path

from dotenv import load_dotenv
from openai import AzureOpenAI
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

load_dotenv()
RESULTS_DIR = Path('..') / 'results'
RESULTS_DIR.mkdir(exist_ok=True)
LABEL_NAMES = {0: 'FAKE', 1: 'REAL'}

client = AzureOpenAI(
    azure_endpoint=os.environ['AZURE_OPENAI_ENDPOINT'],
    api_key=os.environ['AZURE_OPENAI_API_KEY'],
    api_version=os.environ.get('AZURE_OPENAI_API_VERSION', '2024-10-21'),
)
DEPLOYMENT = os.environ['AZURE_OPENAI_CHAT_DEPLOYMENT']

# Smoke test
resp = client.chat.completions.create(
    model=DEPLOYMENT,
    messages=[{'role': 'user', 'content': 'Reply with the single word: ready'}],
)
print('Foundry says:', resp.choices[0].message.content)

Foundry says: ready


## 1. Prepare a test sample

We classify a small random sample (keeps the API cost low). Article text is
truncated to keep each prompt short.

In [2]:
SAMPLE_SIZE = 40
MAX_CHARS = 1500
RANDOM_STATE = 42

ds = load_dataset('GonzaloA/fake_news')
sample = ds['test'].shuffle(seed=RANDOM_STATE).select(range(SAMPLE_SIZE))

def article_text(row):
    title = str(row.get('title') or '')
    text = str(row.get('text') or '')
    return (title + '. ' + text).strip()[:MAX_CHARS]

articles = [article_text(r) for r in sample]
labels = [int(r['label']) for r in sample]
print('Prepared', len(articles), 'articles')

Repo card metadata block was not found. Setting CardData to empty.


Prepared 40 articles


## 2. Prompting strategies

- **Zero-shot:** ask directly for a label.
- **Few-shot:** show a few labelled examples first (in-context learning).
- **Chain-of-thought:** ask the model to reason briefly, then give a final label.

Each strategy ends with a line we can parse: `LABEL: REAL` or `LABEL: FAKE`.

In [3]:
SYSTEM = ('You are a careful fact-checking assistant. You decide whether a news '
          'article is real journalism (REAL) or fabricated / misleading (FAKE).')

FEWSHOT_EXAMPLES = [
    ('Government announces new public transport budget for the next fiscal year, '
     'citing official figures from the finance ministry.', 'REAL'),
    ('SHOCKING: Scientists confirm the moon is made of cheese and governments have '
     'hidden it for decades!!!', 'FAKE'),
    ('Local council approves funding to repair three bridges after an engineering '
     'survey, according to the mayor office.', 'REAL'),
    ('Miracle fruit cures every disease in 24 hours, doctors are furious about this '
     'one secret they do not want you to know.', 'FAKE'),
]

def build_messages(article, strategy):
    if strategy == 'zero_shot':
        user = ('Classify the following news article as REAL or FAKE. '
                'Answer on the last line exactly as: LABEL: REAL or LABEL: FAKE.\n\n'
                'ARTICLE:\n' + article)
        return [{'role': 'system', 'content': SYSTEM},
                {'role': 'user', 'content': user}]
    if strategy == 'few_shot':
        shots = ''
        for ex_text, ex_label in FEWSHOT_EXAMPLES:
            shots += 'ARTICLE: ' + ex_text + '\nLABEL: ' + ex_label + '\n\n'
        user = (shots + 'ARTICLE: ' + article +
                '\nAnswer on the last line exactly as: LABEL: REAL or LABEL: FAKE.')
        return [{'role': 'system', 'content': SYSTEM},
                {'role': 'user', 'content': user}]
    # chain_of_thought
    user = ('Decide whether the article is REAL or FAKE. First write 2-3 short '
            'reasoning sentences about the tone, claims and evidence. Then on the '
            'final line write exactly: LABEL: REAL or LABEL: FAKE.\n\nARTICLE:\n' + article)
    return [{'role': 'system', 'content': SYSTEM},
            {'role': 'user', 'content': user}]

def parse_label(reply):
    m = re.search(r'LABEL:\s*(REAL|FAKE)', reply.upper())
    if m:
        return 1 if m.group(1) == 'REAL' else 0
    up = reply.upper()
    if 'FAKE' in up and 'REAL' not in up:
        return 0
    if 'REAL' in up and 'FAKE' not in up:
        return 1
    return -1  # unparseable

def classify(article, strategy):
    messages = build_messages(article, strategy)
    resp = client.chat.completions.create(model=DEPLOYMENT, messages=messages,
                                          temperature=0, max_tokens=200)
    return parse_label(resp.choices[0].message.content)

## 3. Run each strategy over the sample

This calls the Foundry model once per article per strategy (sample size x 3
short calls). A few articles may be blocked by the content filter; those are
caught and handled in the next step.

In [4]:
STRATEGIES = ['zero_shot', 'few_shot', 'chain_of_thought']
predictions = {s: [] for s in STRATEGIES}

for strategy in STRATEGIES:
    t0 = time.time()
    for i, article in enumerate(articles):
        try:
            pred = classify(article, strategy)
        except Exception as e:
            print('error on', strategy, i, e)
            pred = -1
            time.sleep(2)
        predictions[strategy].append(pred)
    print(f'{strategy}: done in {time.time() - t0:.0f}s')

error on zero_shot 7 Error code: 400 - {'error': {'message': "The response was filtered due to the prompt triggering Azure OpenAI's content management policy. Please modify your prompt and retry. To learn more about our content filtering policies please read our documentation: https://go.microsoft.com/fwlink/?linkid=2198766", 'type': None, 'param': 'prompt', 'code': 'content_filter', 'status': 400, 'innererror': {'code': 'ResponsibleAIPolicyViolation', 'content_filter_result': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': True, 'severity': 'medium'}}}}}


error on zero_shot 35 Error code: 400 - {'error': {'message': "The response was filtered due to the prompt triggering Azure OpenAI's content management policy. Please modify your prompt and retry. To learn more about our content filtering policies please read our documentation: https://go.microsoft.com/fwlink/?linkid=2198766", 'type': None, 'param': 'prompt', 'code': 'content_filter', 'status': 400, 'innererror': {'code': 'ResponsibleAIPolicyViolation', 'content_filter_result': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': True, 'severity': 'medium'}}}}}


zero_shot: done in 45s


error on few_shot 7 Error code: 400 - {'error': {'message': "The response was filtered due to the prompt triggering Azure OpenAI's content management policy. Please modify your prompt and retry. To learn more about our content filtering policies please read our documentation: https://go.microsoft.com/fwlink/?linkid=2198766", 'type': None, 'param': 'prompt', 'code': 'content_filter', 'status': 400, 'innererror': {'code': 'ResponsibleAIPolicyViolation', 'content_filter_result': {'hate': {'filtered': False, 'severity': 'low'}, 'jailbreak': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': True, 'severity': 'medium'}}}}}


error on few_shot 34 Error code: 400 - {'error': {'message': "The response was filtered due to the prompt triggering Azure OpenAI's content management policy. Please modify your prompt and retry. To learn more about our content filtering policies please read our documentation: https://go.microsoft.com/fwlink/?linkid=2198766", 'type': None, 'param': 'prompt', 'code': 'content_filter', 'status': 400, 'innererror': {'code': 'ResponsibleAIPolicyViolation', 'content_filter_result': {'hate': {'filtered': True, 'severity': 'medium'}, 'jailbreak': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'low'}}}}}


error on few_shot 35 Error code: 400 - {'error': {'message': "The response was filtered due to the prompt triggering Azure OpenAI's content management policy. Please modify your prompt and retry. To learn more about our content filtering policies please read our documentation: https://go.microsoft.com/fwlink/?linkid=2198766", 'type': None, 'param': 'prompt', 'code': 'content_filter', 'status': 400, 'innererror': {'code': 'ResponsibleAIPolicyViolation', 'content_filter_result': {'hate': {'filtered': False, 'severity': 'low'}, 'jailbreak': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': True, 'severity': 'medium'}}}}}


few_shot: done in 44s


error on chain_of_thought 34 Error code: 400 - {'error': {'message': "The response was filtered due to the prompt triggering Azure OpenAI's content management policy. Please modify your prompt and retry. To learn more about our content filtering policies please read our documentation: https://go.microsoft.com/fwlink/?linkid=2198766", 'type': None, 'param': 'prompt', 'code': 'content_filter', 'status': 400, 'innererror': {'code': 'ResponsibleAIPolicyViolation', 'content_filter_result': {'hate': {'filtered': True, 'severity': 'medium'}, 'jailbreak': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}}}}


error on chain_of_thought 35 Error code: 400 - {'error': {'message': "The response was filtered due to the prompt triggering Azure OpenAI's content management policy. Please modify your prompt and retry. To learn more about our content filtering policies please read our documentation: https://go.microsoft.com/fwlink/?linkid=2198766", 'type': None, 'param': 'prompt', 'code': 'content_filter', 'status': 400, 'innererror': {'code': 'ResponsibleAIPolicyViolation', 'content_filter_result': {'hate': {'filtered': False, 'severity': 'low'}, 'jailbreak': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': True, 'severity': 'medium'}}}}}


chain_of_thought: done in 72s


## 4. Evaluate each strategy

Some articles are blocked by the hosted model's content filter (real news about
war or crime can trigger the violence / hate filter). Those are reported
separately as *skipped* and excluded from the scores, so the metrics reflect
only the articles the model actually answered. The best strategy is saved for
the comparison notebook and the demo.

In [5]:
import numpy as np
import pandas as pd

llm_results = {}
y_true = np.array(labels)
for strategy in STRATEGIES:
    pred = np.array(predictions[strategy])
    answered = pred >= 0              # articles the model actually classified
    skipped = int((~answered).sum())  # blocked by content filter / unparseable
    yt = y_true[answered]
    yp = pred[answered]
    llm_results[strategy] = {
        'accuracy': accuracy_score(yt, yp),
        'precision': precision_score(yt, yp, zero_division=0),
        'recall': recall_score(yt, yp, zero_division=0),
        'f1': f1_score(yt, yp, zero_division=0),
        'answered': int(answered.sum()),
        'skipped': skipped,
    }

llm_df = pd.DataFrame(llm_results).T.sort_values('f1', ascending=False)
print(llm_df.round(4))
print('\n(skipped = blocked by the content filter, excluded from the scores above)')

best_strategy = llm_df.index[0]
best_metrics = {k: llm_results[best_strategy][k] for k in ['accuracy', 'precision', 'recall', 'f1']}
best_metrics['strategy'] = best_strategy
best_metrics['skipped'] = int(llm_results[best_strategy]['skipped'])

with open(RESULTS_DIR / 'llm_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(best_metrics, f, indent=2)
with open(RESULTS_DIR / 'llm_metrics_all.json', 'w', encoding='utf-8') as f:
    json.dump(llm_results, f, indent=2)
print('Best strategy:', best_strategy)

                  accuracy  precision  recall      f1  answered  skipped
chain_of_thought    0.8684     0.8077     1.0  0.8936      38.0      2.0
zero_shot           0.7632     0.7000     1.0  0.8235      38.0      2.0
few_shot            0.7568     0.7000     1.0  0.8235      37.0      3.0

(skipped = blocked by the content filter, excluded from the scores above)
Best strategy: chain_of_thought
